In [4]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report
)

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce MX450


In [6]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")
test_df = pd.read_csv("../data/processed/test.csv")

In [7]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "label_id"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label_id"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label_id"]],
    preserve_index=False
)

In [8]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [9]:
MAX_LENGTH = 128

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [10]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/4135 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

In [11]:
train_tokenized[0]

{'text': 'Ta-Daaaaa! I am home babe, are you still up ?',
 'label_id': 0,
 'input_ids': [101,
  11937,
  1011,
  4830,
  11057,
  11057,
  999,
  1045,
  2572,
  2188,
  11561,
  1010,
  2024,
  2017,
  2145,
  2039,
  1029,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
train_tokenized

Dataset({
    features: ['text', 'label_id', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4135
})

In [13]:
train_tokenized = train_tokenized.rename_column(
    "label_id",
    "labels"
)

val_tokenized = val_tokenized.rename_column(
    "label_id",
    "labels"
)

test_tokenized = test_tokenized.rename_column(
    "label_id",
    "labels"
)

In [14]:
train_tokenized.column_names

['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "ham",
        1: "spam"
    },
    label2id={
        "ham": 0,
        "spam": 1
    }
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "spam_f1": f1,
        "macro_f1": macro_f1
    }

In [17]:
training_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    fp16=True,

    gradient_checkpointing=True,

    optim="paged_adamw_8bit",

    num_train_epochs=3,

    logging_steps=10,
    save_steps=100,
    eval_steps=100,

    report_to="none",
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [19]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

In [20]:
trainer.train()

Step,Training Loss
10,3.288702
20,1.413823
30,0.193246
40,1.003164
50,1.413642
60,0.603489
70,0.695402
80,1.657730
90,0.754190
100,0.006615


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1551, training_loss=0.41519625827929496, metrics={'train_runtime': 1856.6964, 'train_samples_per_second': 6.681, 'train_steps_per_second': 0.835, 'total_flos': 80013672218268.0, 'train_loss': 0.41519625827929496, 'epoch': 3.0})

In [36]:
val_results = trainer.evaluate(
    eval_dataset=val_tokenized
)

val_results

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,Spam F1,Macro F1
0.000221,0.099300,1551,0.982592,0.938462,0.924242,0.931298,0.960665


{'eval_loss': 0.099300317466259,
 'eval_accuracy': 0.9825918762088974,
 'eval_precision': 0.9384615384615385,
 'eval_recall': 0.9242424242424242,
 'eval_spam_f1': 0.9312977099236641,
 'eval_macro_f1': 0.9606654662575131}

In [37]:
test_results = trainer.evaluate(
    eval_dataset=test_tokenized
)

test_results

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,Spam F1,Macro F1
0.000221,0.032798,1551,0.996132,1.000000,0.969231,0.984375,0.991084


{'eval_loss': 0.032798029482364655,
 'eval_accuracy': 0.9961315280464217,
 'eval_precision': 1.0,
 'eval_recall': 0.9692307692307692,
 'eval_spam_f1': 0.984375,
 'eval_macro_f1': 0.9910837472406181}

In [38]:
predictions = trainer.predict(test_tokenized)

test_logits = predictions.predictions
test_labels = predictions.label_ids

test_predictions = np.argmax(
    test_logits,
    axis=-1
)

In [39]:
test_predictions

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,

In [42]:
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=["ham", "spam"]
    )
)

              precision    recall  f1-score   support

         ham       1.00      1.00      1.00       452
        spam       1.00      0.97      0.98        65

    accuracy                           1.00       517
   macro avg       1.00      0.98      0.99       517
weighted avg       1.00      1.00      1.00       517



In [43]:
distilbert_macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro"
)

print("DistilBERT Macro-F1:", distilbert_macro_f1)

DistilBERT Macro-F1: 0.9910837472406181


In [44]:
results = pd.DataFrame([
    # {
    #     "model": "Majority",
    #     "macro_f1": majority_f1
    # },
    # {
    #     "model": "TF-IDF + Logistic Regression",
    #     "macro_f1": macro_f1
    # },
    {
        "model": "DistilBERT Full Fine-Tuning",
        "macro_f1": distilbert_macro_f1
    }
])

results

,model,macro_f1
0,DistilBERT Full Fine-Tuning,0.991084


In [45]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print(
    "Trainable %:",
    100 * trainable_params / total_params
)

Total parameters: 66955010
Trainable parameters: 66955010
Trainable %: 100.0


In [46]:
print("Device:", device)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable %:", 100 * trainable_params / total_params)

print("\nValidation:")
print(val_results)

print("\nTest:")
print(test_results)

print("\nClassification report:")
print(classification_report(
    test_labels,
    test_predictions,
    target_names=["ham", "spam"]
))

Device: cuda
Total parameters: 66955010
Trainable parameters: 66955010
Trainable %: 100.0

Validation:
{'eval_loss': 0.099300317466259, 'eval_accuracy': 0.9825918762088974, 'eval_precision': 0.9384615384615385, 'eval_recall': 0.9242424242424242, 'eval_spam_f1': 0.9312977099236641, 'eval_macro_f1': 0.9606654662575131}

Test:
{'eval_loss': 0.032798029482364655, 'eval_accuracy': 0.9961315280464217, 'eval_precision': 1.0, 'eval_recall': 0.9692307692307692, 'eval_spam_f1': 0.984375, 'eval_macro_f1': 0.9910837472406181}

Classification report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00       452
        spam       1.00      0.97      0.98        65

    accuracy                           1.00       517
   macro avg       1.00      0.98      0.99       517
weighted avg       1.00      1.00      1.00       517



In [47]:
test_results

{'eval_loss': 0.032798029482364655,
 'eval_accuracy': 0.9961315280464217,
 'eval_precision': 1.0,
 'eval_recall': 0.9692307692307692,
 'eval_spam_f1': 0.984375,
 'eval_macro_f1': 0.9910837472406181}

In [48]:
test_results = {
    key.removeprefix("eval_"): value
    for key, value in test_results.items()
}

In [49]:
test_results['model'] = "full_fine_tune"

In [50]:

fine_tune = pd.DataFrame([test_results])
fine_tune.to_csv("full_fine_tune.csv")

In [51]:
baseline= pd.read_csv("baseline_results.csv")
tfidf_baseline = pd.read_csv("baseline_results_tfidf.csv")
fine_tune_baseline = pd.read_csv("full_fine_tune.csv")

In [52]:
all_results = pd.concat(
    [baseline, tfidf_baseline, fine_tune_baseline],
    ignore_index=True
)

all_results.to_csv("results.csv")


In [55]:
from pathlib import Path

MODEL_DIR = Path("models/distilbert-full-ft")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print(f"Model saved to: {MODEL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: models/distilbert-full-ft


In [58]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

backup_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

backup_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR
)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.


In [59]:
print("Parameters:", sum(p.numel() for p in backup_model.parameters()))

Parameters: 66955010


In [60]:
for param in backup_model.parameters():
    param.requires_grad = False

In [61]:
trainable_params = sum(
    p.numel()
    for p in backup_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Trainable parameters: 0
